# Lab 1: A Feed-Forward Neural Network in PyTorch

**Time:** about 75 minutes

Yesterday XGBoost (eXtreme Gradient Boosting) ranked claims well enough to catch a good share of irregular claims with
only 5% of claims reviewed. Today's question is simple, and it's one you'll be asked in real projects:

> *Would a neural network do better?*

You'll build one in PyTorch, piece by piece, so that you understand every part of how a neural network learns. Then
you'll compare it honestly with the models from Day 2.

The network is deliberately small. This lab is about understanding the mechanics, not about building the biggest
model. (You'll meet much larger networks, called transformers, on Day 6.)

### What is a neural network, in one paragraph?

A neural network is a model made of layers of simple calculations. Each layer takes a list of numbers, multiplies them
by **weights**, adds them up, and passes the result through a simple function. Stack a few layers and the network can
learn complicated patterns. At the start the weights are random, so the predictions are useless. **Training** means
showing the network many examples with known answers and nudging the weights, a tiny bit at a time, so the predictions
get closer to the right answers. Everything in this lab is about how that nudging works.

### Words you'll meet in this lab

| Word | Plain meaning |
|---|---|
| **PyTorch** | a Python library for building and training neural networks |
| **tensor** | PyTorch's version of an array of numbers (like a numpy array), which can also live on a GPU |
| **GPU** | graphics processing unit: a chip that does many calculations at once, so training is much faster |
| **weight** | a number inside the network that is learned during training |
| **layer** | one step of calculation in the network: inputs × weights, added up |
| **loss** | one number saying how wrong the predictions are; training tries to make it small |
| **gradient** | for each weight, which direction (up or down) would reduce the loss, and how strongly |
| **backpropagation** | the method that works out the gradient for every weight in the network |
| **optimiser** | the part that actually changes the weights, using the gradients |
| **epoch** | one full pass over all the training examples |
| **mini-batch** | a small group of examples (here 128) processed together in one training step |
| **validation set** | data the network doesn't learn from, used to decide when to stop training |
| **overfitting** | when a model memorises its training examples instead of learning patterns that work on new data |

## Where this lab fits

**What you've done so far.** On Day 1 you cleaned the Fund's data and built a pipeline. On Day 2 you built a claims
risk score: logistic regression, a decision tree and XGBoost, tested fairly against the review team's 5% capacity,
across four years, for fairness and for leakage. XGBoost won, and you saved it with a model card.

**What you'll do now.** You'll build a neural network for the same job, from the smallest piece upwards: one neuron
learning by hand, then tensors, then a full network with a training loop you write yourself. Then you'll compare it
with Day 2's models on exactly the same claims.

**Why it matters.** Neural networks are behind almost everything you'll meet from Day 6 onwards: the BERT model that
routes complaints, the large language models on Days 7-9. They all learn with the same loop you'll write today. And
you'll be asked "wouldn't deep learning do better?" in real projects, so you need to know how to answer it honestly.

**What happens next.** Lab 2 turns to a completely different problem: data too big to fit in memory. Lab 3 makes the
whole chain reproducible, so anyone can rebuild exactly the same features and results later.

### Theory: how a neural network differs from Day 2's models

All of Day 2's models, and today's network, learn a way of turning features into a score. The difference is in what
they can learn and how:

| | Logistic regression | Boosted trees (XGBoost) | Neural network |
|---|---|---|---|
| What it learns | one weight per feature | hundreds of yes/no question trees | layers of weights, passed through bends (ReLU) |
| Patterns it can find | straight-line effects only | interactions and thresholds | almost any pattern, given enough data |
| How it learns | adjust weights to reduce the loss | add trees one at a time to fix errors | adjust *all* weights together, by gradient descent |
| Data it needs | little | moderate | a lot, especially for complex patterns |
| Settings to choose | very few | a handful | many: layers, sizes, learning rate, dropout, epochs, batch size |

The big strength of neural networks is that their middle (hidden) layers **learn their own features**. A network
trained on images learns, by itself, to detect edges, then shapes, then objects. That's why they dominate for images,
sound and text, where hand-made features are hard to write. For a table where people have already engineered good
features, like your claims table, that strength matters much less. Keep that in mind when you see the comparison in
Section 7.

## 0. Set up today's environment (run this cell first)

You don't need to create a folder or an environment by hand today. The cell below does it for you:

1. It finds the environment you created on Day 1 (`DAY1/Labs/work/fund-ai/.venv`). If it isn't there, it creates it.
2. It installs today's libraries into it from ``requirements.txt``. The first time takes a few minutes; after that, seconds.
3. If this machine has an NVIDIA GPU (graphics processing unit), it switches PyTorch to its GPU build, so training runs much faster.
4. It connects the environment to Jupyter as **Python (fund-ai)**, and tells you if this notebook isn't using it yet.

It's safe to run again at any time. If it prints instructions to switch the kernel, follow them and run it once more.

You don't need to read the code in this cell. It's housekeeping, not part of today's lesson.

In [ ]:
# ---- Setup for Day 3. Safe to run more than once. -------------------------------------------------------------
COURSE_ENV = None        # None = use the Day 1 environment. Or set a folder, e.g. Path.home() / "fund-ai-venv"
REQUIREMENTS = "requirements.txt"
EXTRAS = []

import os, shutil, subprocess, sys
from pathlib import Path

labs = Path.cwd()
root = next((p for p in [labs, *labs.parents] if (p / "DAY1").is_dir()), None)
if root is None:
    raise RuntimeError("Open this notebook from inside the course folder (the one that contains DAY1, DAY2, ...).")
env = Path(COURSE_ENV) if COURSE_ENV else root / "DAY1" / "Labs" / "work" / "fund-ai" / ".venv"
py = env / ("Scripts/python.exe" if os.name == "nt" else "bin/python")


def run(*cmd, quiet=False):
    """Run a command; show its output only if it fails."""
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    if r.returncode != 0 and not quiet:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if "path" in (r.stderr + r.stdout).lower() and os.name == "nt":
            print("\nIf the error mentions a path that is too long: Windows 'long paths' are switched off on this machine.\n"
                  "Ask ICT to enable them, or set COURSE_ENV above to a short folder, e.g. Path.home() / 'fund-ai-venv'.")
        raise RuntimeError("A setup step failed. The details are printed above.")
    return r


# 1. The course environment
if py.exists():
    print(f"- course environment found: {env}")
else:
    print(f"- creating the course environment (first time only): {env}")
    run(getattr(sys, "_base_executable", sys.executable), "-m", "venv", env)
    run(py, "-m", "pip", "install", "-q", "--upgrade", "pip")

# 2. Today's libraries
req = labs / REQUIREMENTS
print(f"- installing today's libraries from {REQUIREMENTS} ...")
run(py, "-m", "pip", "install", "-q", "-r", req)

# 3. Use the GPU if this machine has an NVIDIA one
if "torch" in req.read_text() and shutil.which("nvidia-smi"):
    if run(py, "-c", "import torch; print(torch.cuda.is_available())").stdout.strip() != "True":
        version = run(py, "-c", "import torch; print(torch.__version__.split('+')[0])").stdout.strip()
        print(f"- NVIDIA GPU found: switching PyTorch {version} to its GPU build (about 2.5 GB, first time only) ...")
        run(py, "-m", "pip", "uninstall", "-y", "-q", "torch")
        run(py, "-m", "pip", "install", "-q", f"torch=={version}", "--index-url", "https://download.pytorch.org/whl/cu124",
            "--extra-index-url", "https://pypi.org/simple")
    print("- PyTorch will use the GPU")

# 4. Anything else today needs
for extra in EXTRAS:
    kind, name = extra.split(":", 1)
    if kind == "spacy" and run(py, "-c", f"import {name}", quiet=True).returncode != 0:
        print(f"- downloading the spaCy model {name} ...")
        run(py, "-m", "spacy", "download", name)
    if kind == "ollama":
        import json, urllib.request
        try:
            tags = json.load(urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3))["models"]
            have = {m["name"] for m in tags}
            wanted = [m for m in name.split(",") if m]
            missing = [m for m in wanted if m not in have]
            print(f"- Ollama is running with {len(have)} model(s)" + (f"; still to pull: {', '.join(missing)}" if missing else ""))
            for m in missing:
                print(f"    in a terminal: ollama pull {m}")
        except OSError:
            print("- Ollama isn't running. Start the Ollama app (or run `ollama serve` in a terminal) before the LLM parts.")

# 5. Connect the environment to Jupyter
run(py, "-m", "ipykernel", "install", "--user", "--name", "fund-ai", "--display-name", "Python (fund-ai)")

# 6. Is this notebook using it?
if Path(sys.prefix).resolve() == env.resolve():
    print("\nReady. This notebook is running in the course environment. Carry on with the next cell.")
else:
    print("\nAlmost there. This notebook is running a different Python:")
    print(f"    {sys.executable}")
    print("Switch it to the course environment, then run this cell once more:")
    print("    1. Click the kernel name in the top right of the notebook, then 'Select Another Kernel'.")
    print("    2. Choose 'Jupyter Kernel' -> 'Python (fund-ai)'.")
    print("       (Or press Ctrl+Shift+P, run 'Python: Select Interpreter', choose 'Enter interpreter path...'")
    print("        and paste the path below, the same way you did on Day 1.)")
    print(f"    {py}")

## Imports

This cell loads the general-purpose libraries you already know from Days 1 and 2: `numpy` and `pandas` for numbers
and tables, `matplotlib` for charts, `json` and `joblib` to save files, and `time` to measure how long training takes.

In [ ]:
import os
import json                        # saves the network's settings to a text file
import time                        # measures how long training takes
import warnings
from pathlib import Path

# Windows 11 no longer has a tool ("wmic") that some libraries use to count CPU cores; they print a long, harmless
# error about it. Telling them how many cores to use skips that check. The second line hides library notices.
os.environ["LOKY_MAX_CPU_COUNT"] = str(os.cpu_count() - 1)
warnings.filterwarnings("ignore")

import joblib                      # saves the data-preparation steps to a file
import numpy as np                 # arrays of numbers
import pandas as pd                # tables
import matplotlib.pyplot as plt    # charts

Now the new library, **PyTorch**, which is imported as `torch`. Three parts of it are used:

- `torch` itself: tensors and the maths on them;
- `torch.nn` ("neural networks"): ready-made layers and loss functions to build a network from;
- `TensorDataset` and `DataLoader`: tools that feed the training data to the network in small groups.

The scikit-learn and XGBoost imports are for preparing the data (as on Day 2) and for the models you'll compare against.

In [ ]:
# PyTorch: the deep-learning library.
import torch
from torch import nn                                      # building blocks for networks (layers, losses)
from torch.utils.data import TensorDataset, DataLoader    # feed the data in mini-batches

# scikit-learn, for preparing the data and for the models to compare against.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

Two settings before we start:

- **The random seed.** A new network starts with random weights, and training shuffles the data randomly. Setting a
  *seed* (any fixed number) makes that randomness the same every time you run the notebook, so your numbers match your
  neighbour's and you can repeat an experiment exactly.
- **The device.** PyTorch can calculate on the CPU (the computer's normal processor) or on an NVIDIA GPU, which is much
  faster for neural networks. `torch.cuda.is_available()` says whether a GPU is usable ("CUDA" is NVIDIA's name for
  its GPU programming system). Everything will be sent to whichever `device` you have.

In [ ]:
# Fix the random seeds so results are repeatable.
SEED = 42
torch.manual_seed(SEED)       # PyTorch's randomness (starting weights, shuffling)
np.random.seed(SEED)          # numpy's randomness

# Use the GPU if there is one, otherwise the CPU.
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
print("PyTorch", torch.__version__, "| running on", device)

> **Where we are.** PyTorch is loaded, the random seed is fixed, and you know whether you're training on a CPU or a
> GPU. Before building a network, you'll meet its basic ingredient (the tensor) and watch a single neuron learn.

## 1. Tensors in two minutes

A **tensor** is PyTorch's version of a numpy array: a list (or table, or cube) of numbers. You can add, multiply and
compare tensors just like numpy arrays. Tensors have two extra abilities that neural networks need:

1. they can be moved to a GPU, where calculations run much faster;
2. they can **remember how they were calculated**, so PyTorch can work backwards later and figure out how each input
   affected the result. That's what makes training possible (you'll see it in a moment).

Every tensor has a **dtype** (the type of number, e.g. `float32` = decimal numbers stored in 32 bits) and a **shape**
(how many numbers along each direction; `[3]` means a list of three numbers).

In [ ]:
# A tensor of three wages. The dots make them decimals (float), which is what networks work with.
wages = torch.tensor([450_000., 800_000., 1_250_000.])

# Maths on a tensor works on every number at once, like numpy: here, 20% of each wage.
print(wages * 0.20)

# The type of number, and the shape: torch.Size([3]) = a list of 3 numbers.
print("type:", wages.dtype, " shape:", wages.shape)

You'll often prepare data with numpy or pandas and then hand it to PyTorch. `torch.from_numpy` turns a numpy array
into a tensor, and `.numpy()` turns a tensor back into a numpy array.

In [ ]:
# Converting between numpy and PyTorch.
numpy_array = np.array([1.0, 2.0])
as_tensor = torch.from_numpy(numpy_array)      # numpy -> PyTorch
back_to_numpy = as_tensor.numpy()              # PyTorch -> numpy
print(as_tensor, back_to_numpy)

### How does a network learn? One neuron, step by step

Before building a whole network, look at the single idea that makes learning possible, using one **neuron**.

A neuron does a tiny calculation: it takes an input `x`, multiplies it by a **weight** `w`, adds a **bias** `b`, and
squeezes the result through the **sigmoid** function, which turns any number into a value between 0 and 1. That value
is the neuron's prediction, a probability:

```
prediction = sigmoid(w × x + b)
```

Suppose the correct answer for this example is `y = 1` (e.g. "irregular"). We measure how wrong the prediction is with
a **loss**. The loss used for yes/no questions is called **binary cross-entropy**: it's close to 0 when the prediction is
close to the right answer, and grows quickly as the prediction gets further away.

The key question for learning is: *if I nudge `w` up a little, does the loss go up or down, and by how much?* That number
is the **gradient** of the loss with respect to `w`. Working out the gradients is called **backpropagation**. PyTorch
does it automatically when you call `loss.backward()`. Below you'll check PyTorch's answer against the formula you'd
get by doing the calculus by hand.

First, set up the example and the neuron's two numbers. `requires_grad=True` tells PyTorch "this is a number I want to
learn; keep track of how the loss depends on it".

In [ ]:
# One training example: input 2.0, correct answer 1.
x = torch.tensor(2.0)
y = torch.tensor(1.0)

# The neuron's two learnable numbers, with starting values chosen by hand.
# requires_grad=True asks PyTorch to track how the loss depends on them.
w = torch.tensor(0.5, requires_grad=True)
b = torch.tensor(-1.0, requires_grad=True)

The **forward pass** means: run the input through the calculation to get a prediction, then measure the loss.
With `w = 0.5`, `x = 2` and `b = -1`, the neuron computes `sigmoid(0.5 × 2 - 1) = sigmoid(0) = 0.5`: a 50/50 guess. The
correct answer is 1, so the loss is well above zero.

In [ ]:
# Forward pass: the prediction, a probability between 0 and 1.
p = torch.sigmoid(w * x + b)

# The loss (binary cross-entropy): small when p is close to y, large when it's far away.
loss = -(y * torch.log(p) + (1 - y) * torch.log(1 - p))

# .item() turns a one-number tensor into a plain Python number, for printing.
print("prediction:", round(p.item(), 3))
print("loss:      ", round(loss.item(), 3))

The **backward pass**: `loss.backward()` works out the gradient for every number that had `requires_grad=True`, and
stores it in `.grad`. For this particular loss the calculus gives a simple answer, `(p - y) × x` for the weight and
`(p - y)` for the bias, so you can check PyTorch's work.

A **negative** gradient means "increasing this number would reduce the loss", so the weight should go up.

In [ ]:
# Backward pass: PyTorch works out the gradients and stores them in w.grad and b.grad.
loss.backward()
print("autograd: dL/dw =", round(w.grad.item(), 4), " dL/db =", round(b.grad.item(), 4))

# The same gradients by hand, from the formula for this loss: dL/dw = (p - y) * x, dL/db = (p - y).
print("by hand:  dL/dw =", round(((p - y) * x).item(), 4), " dL/db =", round((p - y).item(), 4))

They match. A network with thousands of weights works exactly the same way: PyTorch applies this calculation layer by
layer, from the output back to the input (that's why it's called *back*-propagation).

Now **learn**: move each number a small step in the direction that reduces the loss. This is called **gradient
descent**:

```
new weight = old weight - learning_rate × gradient
```

The **learning rate** controls the size of the step. Too big and training jumps around; too small and it takes forever.
After one step, the prediction should be closer to the correct answer, 1.

In [ ]:
learning_rate = 0.5

# torch.no_grad(): we're changing the weights by hand, so don't track this step as part of a calculation.
with torch.no_grad():
    w -= learning_rate * w.grad      # step the weight against its gradient
    b -= learning_rate * b.grad      # same for the bias

# Run the forward pass again with the updated numbers.
new_p = torch.sigmoid(w * x + b)
print("new prediction:", round(new_p.item(), 3), "(closer to the label 1)")

That's the whole idea of training: **predict, measure the loss, work out the gradients, take a small step**, and repeat
thousands of times over many examples. Everything else in this lab is that loop, made practical.

## 2. The data, prepared exactly as on Day 2

You'll use the same claims, the same extra columns (features) and the same split by date as Day 2, so the comparison
with Day 2's models is fair.

There's one addition: a **validation set**. A neural network trains in many rounds, and after each round you need to
know whether it's still improving on claims it hasn't seen. You can't use the 2025 test set for that: if you keep
looking at the test set to make decisions, it's no longer a fair final test. So the 2024 claims become the validation
set:

```
2019 ─────────── 2023 │ 2024 │ 2025
      train           │ val  │ test
```

- **train** (2019-2023): the network learns from these;
- **validation** (2024): checked after every round, to decide when to stop;
- **test** (2025): looked at only once, at the very end, for the honest comparison.

First read the claims and add the same features as Day 2. Each one turns raw data into a warning sign a claims officer
would recognise.

> **Where we are.** You've watched one neuron learn from one example: predict, measure the loss, get the gradients,
> take a step. The rest of the lab scales that up: thousands of claims instead of one, thousands of weights instead
> of two, and a loop that repeats the step many times. First, the data.

In [ ]:
claims = pd.read_csv("claims_review_history.csv", parse_dates=["claim_date"])

# The same engineered features as Day 2.
# How big is the claim compared with the member's own wage?
claims["amount_to_wage_ratio"] = claims["amount_claimed_tzs"] / claims["avg_wage_last_6m_tzs"]
# Were the bank details ever changed? Changed in the last 60 days? (1 = yes, 0 = no)
claims["bank_ever_changed"] = claims["bank_details_changed_days_ago"].notna().astype(int)
claims["bank_changed_60d"] = (claims["bank_details_changed_days_ago"] <= 60).astype(int)
# Has the member claimed before?
claims["has_previous_claim"] = (claims["previous_claims"] > 0).astype(int)
# Empty means "never": use 9999 days ("a very long time ago").
claims["days_since_last_claim"] = claims["days_since_last_claim"].fillna(9999)
claims["bank_details_changed_days_ago"] = claims["bank_details_changed_days_ago"].fillna(9999)
# Squeeze the very wide range of claim amounts onto a smaller scale.
claims["log_amount"] = np.log1p(claims["amount_claimed_tzs"])

# The columns the model uses: numbers, and text categories.
NUMERIC = [
    "member_age", "months_contributed", "months_contributed_last_12", "log_amount", "amount_to_wage_ratio",
    "employer_arrears_months_12m", "days_since_registration", "previous_claims", "days_since_last_claim",
    "documents_missing_at_lodging", "bank_details_changed_days_ago", "bank_ever_changed", "bank_changed_60d",
    "has_previous_claim",
]
CATEGORICAL = ["benefit_type", "channel", "employer_sector", "gender"]
FEATURES = NUMERIC + CATEGORICAL

Now split by date into the three periods, and check that each has a similar share of irregular claims. If one period
had far more or fewer problem claims, the comparison would be harder to read.

In [ ]:
# Three periods: up to 2023 = train, 2024 = validation, 2025 = test.
train = claims[claims["claim_date"] < "2024-01-01"]
val = claims[(claims["claim_date"] >= "2024-01-01") & (claims["claim_date"] < "2025-01-01")]
test = claims[claims["claim_date"] >= "2025-01-01"]

# How many claims in each, and what percentage were irregular?
print("train:", len(train), "claims, irregular", round(train["irregular"].mean() * 100, 1), "%")
print("val:  ", len(val), "claims, irregular", round(val["irregular"].mean() * 100, 1), "%")
print("test: ", len(test), "claims, irregular", round(test["irregular"].mean() * 100, 1), "%")

### Preparing the inputs: why scaling matters even more here

A neural network can only work with numbers, with no gaps, so the same three preparation steps as Day 2 are needed:

1. **fill the gaps** (with the middle value for numbers, the most common value for categories);
2. **scale the numbers** so each one has an average of 0 and a typical spread of 1;
3. **one-hot encode** the categories (one 0/1 column per value).

Scaling matters much more for neural networks than for tree models. Training adjusts weights in small steps. If one
input is in the thousands (days since registration) and another is 0 or 1 (a flag), the big input dominates the early
steps and the small one is almost ignored. After scaling, every input starts on an equal footing.

As always, the preparation **learns** its values (medians, averages, category lists) from the training data only, with
`fit_transform`. Validation and test are only **transformed** with those same values, so no information from them leaks
into training.

In [ ]:
# The same preparation as Day 2: fill gaps, scale the numbers, one-hot encode the categories.
# sparse_output=False gives an ordinary table of numbers, which PyTorch needs.
prep = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), CATEGORICAL),
])

# fit_transform on the TRAINING data: learn the medians and scaling, then apply them.
X_tr = prep.fit_transform(train[FEATURES])

# transform (no fit) on validation and test: apply the same values, don't re-learn them.
X_va = prep.transform(val[FEATURES])
X_te = prep.transform(test[FEATURES])

# 18 original columns become more after one-hot encoding (one column per category value).
print("input features after preparation:", X_tr.shape[1])

### Why `float32` here?

Computers store decimal numbers with a fixed number of digits. `float64` keeps about 15 significant digits; `float32`
keeps about 7 and uses half the memory.

On Day 1 you kept money as `float64`, because a shilling amount needs to be exact. Neural networks are the opposite
case: their weights don't need 15 digits of precision, and `float32` is both smaller and what GPUs are fastest at. So
the arrays are converted with `.astype(np.float32)` before they become tensors. (The amounts have already been turned
into logs and scaled, so nothing important is lost.)

In [ ]:
# Inputs as float32.
X_tr = X_tr.astype(np.float32)
X_va = X_va.astype(np.float32)
X_te = X_te.astype(np.float32)

# Labels (0 or 1) as float32 too: the loss function expects decimals.
y_tr = train["irregular"].to_numpy().astype(np.float32)
y_va = val["irregular"].to_numpy().astype(np.float32)
y_te = test["irregular"].to_numpy().astype(np.float32)

# The type, the shape of the inputs (rows, columns), and the number of labels.
print(X_tr.dtype, X_tr.shape, y_tr.shape)

> **Where we are.** The claims are split into train (2019-2023), validation (2024) and test (2025), prepared with
> preparation steps learned from the training claims only, and converted to `float32`. Next you'll set up how those
> claims are handed to the network during training.

## 3. Feeding the data to the network: Dataset and DataLoader

A network doesn't learn from all 5,000 training claims in one go. It learns from **mini-batches**: small groups of
examples (here 128 claims). For each mini-batch it makes predictions, measures the loss, and takes one small step.
Learning in mini-batches is:

- **faster** than one claim at a time, because the computer processes 128 claims in parallel;
- **more stable** than one claim at a time, because each step is based on an average over 128 examples;
- **more memory-friendly** than all claims at once, which matters with big datasets.

PyTorch splits this job into two tools:

- a **Dataset** knows how many examples there are and can hand over example number *i* (its inputs and its label).
  For data already in arrays, `TensorDataset` does that for you;
- a **DataLoader** takes a Dataset and serves it as mini-batches, in a new random order each epoch. Shuffling stops the
  network learning anything from the order the claims happen to be in.

In [ ]:
# Pair each claim's features with its label: example i = (features of claim i, label of claim i).
train_dataset = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr))
print("examples in the training set:", len(train_dataset))

# Example number 0: (features, label).
features_0, label_0 = train_dataset[0]
print("features of example 0:", features_0.shape, " label:", label_0)

In [ ]:
# Mini-batches of 128 claims, in a new random order every epoch.
BATCH = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH, shuffle=True)

# A DataLoader is looped over. Take just the first batch to see its shape: 128 rows of features, 128 labels.
for xb, yb in train_loader:
    print("one batch:", xb.shape, yb.shape)
    break          # stop after the first batch

The validation set is small (about 1,200 claims), so there's no need to split it into batches: it'll be scored in one
go after each epoch. Convert it to tensors once, and move it to the device (GPU or CPU) where the network will be.

In [ ]:
# The validation set as two tensors. .to(device) moves them to the GPU, if there is one.
X_va_t = torch.from_numpy(X_va).to(device)
y_va_t = torch.from_numpy(y_va).to(device)

> **Where we are.** The training claims are wrapped in a `DataLoader` that serves them as shuffled mini-batches of 128,
> and the validation claims are ready as tensors. Now you'll design the network itself.

## 4. The model

Here's the network you'll build. Each box is a layer; the numbers underneath are how many values come out of it:

```
input (≈39 features) ─► Linear ─► ReLU ─► Dropout ─► Linear ─► ReLU ─► Linear ─► one number (logit)
                        64                           32                   1
```

What each kind of layer does:

- **`Linear`**: the layer that holds weights. `nn.Linear(39, 64)` takes 39 numbers in and produces 64 numbers out; each
  output is a weighted sum of all 39 inputs plus a bias. These weights are what training learns.
- **`ReLU`** (rectified linear unit): replaces every negative number with 0 and keeps positive numbers as they are. It
  sounds trivial, but it's essential. Without it, stacking several `Linear` layers would be no more powerful than a
  single one (a weighted sum of weighted sums is still just a weighted sum). ReLU lets the network learn bends and
  combinations, such as "a recent bank change matters *more* when the claim is online".
- **`Dropout(0.2)`**: during training, switches off a random 20% of the values each time. The network can't rely on any
  one path, so it's harder for it to memorise individual claims. (Dropout is switched off when making real predictions.)
- **The last `Linear(32, 1)`** outputs one number per claim, called a **logit**: a raw score where higher means "more
  likely irregular". To turn it into a probability between 0 and 1 you'd apply the sigmoid function; the loss function
  in Section 5 does that internally, because it's more accurate that way.

The 64 and 32 in the middle are the **hidden layers**: "hidden" simply because you never see their values directly.

`nn.Sequential` builds a network from a list of layers that are applied in order, top to bottom.

In [ ]:
n_inputs = X_tr.shape[1]     # how many numbers describe each claim after preparation

model = nn.Sequential(
    nn.Linear(n_inputs, 64),   # input -> 64 neurons
    nn.ReLU(),                 # negatives become 0
    nn.Dropout(0.2),           # switch off 20% of them at random while training
    nn.Linear(64, 32),         # 64 -> 32 neurons
    nn.ReLU(),
    nn.Linear(32, 1),          # 32 -> one output score per claim
)
model = model.to(device)       # put the network's weights on the GPU (or CPU)
print(model)

How many weights will be learned? Each `Linear` layer has one weight for every input-output pair, plus one bias per
output. For the first layer that's `39 × 64 + 64`. Counting them shows the size of the model. (For comparison: the
language models on Day 7 have *billions*.)

In [ ]:
# Count the weights the network will learn. .numel() = number of elements in one weight tensor.
n_weights = 0
for parameter in model.parameters():
    n_weights = n_weights + parameter.numel()
print("trainable parameters:", n_weights)

Before training, try the network on the batch from earlier, to check the shapes fit together. The output has shape
`(128, 1)`: 128 claims, one score each. The labels have shape `(128,)`. `.squeeze(1)` removes the extra dimension so the
two match. (The scores are meaningless for now: the weights are still random.)

In [ ]:
# Try it on one batch. no_grad(): we're only looking, not training.
with torch.no_grad():
    out = model(xb.to(device))
print(out.shape, "->", out.squeeze(1).shape)

> **Where we are.** The network is built, it has a few thousand weights (all random for now), and its shapes fit the
> data. It can make predictions, but they're meaningless. To learn, it needs a way to measure its mistakes and a way
> to change its weights.

### Theory: what the loss is really measuring

**Binary cross-entropy** punishes a prediction according to how confident and how wrong it is. For a claim that
really is irregular (answer 1):

| Predicted probability | 0.99 | 0.9 | 0.5 | 0.1 | 0.01 |
|---|---|---|---|---|---|
| Loss | 0.01 | 0.11 | 0.69 | 2.30 | 4.61 |

A confident right answer costs almost nothing. A 50/50 guess costs a moderate amount. A confident **wrong** answer costs
a lot, and the cost grows very fast as the prediction approaches 0. That steep growth is what drives learning: the
gradients are largest exactly where the network is most wrong.

The loss for a mini-batch is the average over its 128 claims. Training is the search for weights that make this average
as small as possible across all the training claims.

## 5. Loss and optimiser

Two more pieces are needed before training.

**The loss function** measures how wrong the predictions are. For yes/no questions the standard choice is **binary
cross-entropy** (the same loss you calculated by hand in Section 1). `BCEWithLogitsLoss` takes the network's raw scores
(logits), applies the sigmoid internally, and computes the loss.

There's a catch. Only about 1 in 10 claims is irregular. A network could get a fairly low loss by predicting "regular"
for everything, and never learn to spot problems. `pos_weight` fixes that: it makes every mistake on an irregular claim
count about 10 times as much. It does the same job as `class_weight="balanced"` did for the Day 2 models.

**The optimiser** changes the weights after each mini-batch, using the gradients. Section 1 used the simplest rule
(`weight -= learning_rate × gradient`). **Adam** is an improved version used almost everywhere: it automatically adjusts
the step size for each weight, which makes training faster and more reliable.

- `lr=1e-3` (0.001) is the learning rate: the size of the steps.
- `weight_decay=1e-4` gently pulls every weight towards zero, which discourages the network from relying on any one input
  too heavily. It's another guard against memorising the training data.

In [ ]:
# About 10 regular claims for every irregular one, so a missed irregular claim should cost about 10x more.
ratio = (y_tr == 0).sum() / (y_tr == 1).sum()
pos_weight = torch.tensor(ratio, device=device)
print("pos_weight:", round(pos_weight.item(), 2))

# The loss: binary cross-entropy on the raw scores, with irregular claims weighted up.
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# The optimiser: Adam, updating all the network's weights.
optimiser = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

> **Where we are.** You have every piece: data in mini-batches, a network, a loss that measures mistakes (with extra
> weight on irregular claims), and an optimiser that changes the weights. The training loop puts them together.

### Theory: three guards against memorising

A network with thousands of weights can memorise 5,000 training claims almost perfectly. That's **overfitting**: it
learns the noise in the training claims instead of patterns that hold for new ones. This lab uses three guards against
it, each working in a different way. Together they're called **regularisation**:

| Guard | How it works | Where it is |
|---|---|---|
| **Dropout** | randomly switches off 20% of values during training, so no single path can carry a memorised answer | in the model (Section 4) |
| **Weight decay** | gently pulls all weights towards zero, so the network prefers simple explanations | in the optimiser (Section 5) |
| **Early stopping** | watches the validation set and stops when it stops improving | in the training loop (below) |

Early stopping is the most important of the three, and it's the reason the validation set exists. Training loss alone
always looks good; only data the network isn't learning from can tell you when it has started memorising.

## 6. The training loop

Every deep-learning project, from this small network to the largest language models, trains with the same loop. Read
the next cell slowly; it's the heart of the lab.

For each **epoch** (one pass over all the training claims):

1. **Learn from every mini-batch.** For each batch of 128 claims:
   - **forward pass**: the network predicts a score for each claim;
   - **loss**: how wrong were the predictions?
   - **zero the gradients**: PyTorch adds new gradients to old ones, so clear the previous batch's first;
   - **backward pass**: work out the gradient for every weight (backpropagation);
   - **optimiser step**: nudge every weight a little in the direction that reduces the loss.
2. **Check the validation set**, *without* learning from it. How good is the network on claims it hasn't trained on?
   We measure PR-AUC (area under the precision-recall curve): a score from 0 to 1 for how well irregular claims are
   ranked at the top, which is what matters when problem cases are rare.
3. **Early stopping.** Keep a copy of the weights from the best epoch so far. If validation hasn't improved for
   `PATIENCE` epochs in a row, stop: further training would only make the network memorise the training claims.

Two switches appear in the loop: `model.train()` turns dropout on (for learning), and `model.eval()` turns it off (for
checking).

First the settings, and some empty containers to record what happens.

In [ ]:
EPOCHS = 60        # at most 60 passes over the training data
PATIENCE = 8       # stop if validation hasn't improved for 8 epochs in a row

history = []                       # one row of numbers per epoch, for the charts
best_score = -1                    # best validation PR-AUC so far (starts below any real score)
best_weights = None                # a copy of the weights at that point
epochs_without_improvement = 0     # how many epochs since the last improvement

start = time.perf_counter()        # start the clock

Now the loop itself. It prints one line when early stopping kicks in, and the total training time.

In [ ]:
for epoch in range(1, EPOCHS + 1):

    # ---- 1. Train on every mini-batch ----
    model.train()                                  # switches dropout on
    total_loss = 0.0
    for xb, yb in train_loader:
        # Move this batch to the same device as the network.
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb).squeeze(1)              # forward pass: one score per claim
        loss = loss_fn(logits, yb)                 # how wrong?

        optimiser.zero_grad()                      # clear the gradients from the last batch
        loss.backward()                            # backpropagation: a gradient for every weight
        optimiser.step()                           # update the weights

        # Add up the loss over the epoch (times the batch size, so we can average properly below).
        total_loss = total_loss + loss.item() * len(yb)
    train_loss = total_loss / len(train_dataset)   # average training loss for this epoch

    # ---- 2. Check the validation set (no learning) ----
    model.eval()                                   # switches dropout off
    with torch.no_grad():                          # no gradients needed: we're only measuring
        val_logits = model(X_va_t).squeeze(1)
        val_loss = loss_fn(val_logits, y_va_t).item()
        val_scores = torch.sigmoid(val_logits).cpu().numpy()   # raw scores -> probabilities, back to numpy
    val_pr_auc = average_precision_score(y_va, val_scores)

    # Record this epoch's numbers.
    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_pr_auc": val_pr_auc})

    # ---- 3. Early stopping ----
    if val_pr_auc > best_score:
        # A new best: remember the score and a copy of the weights.
        best_score = val_pr_auc
        # state_dict() holds all the weights. Copy each one, so later training doesn't change the saved copy.
        best_weights = {}
        for name, value in model.state_dict().items():
            best_weights[name] = value.clone()
        epochs_without_improvement = 0
    else:
        # No improvement this epoch. Stop if that's been the case for PATIENCE epochs.
        epochs_without_improvement = epochs_without_improvement + 1
        if epochs_without_improvement >= PATIENCE:
            print("early stop at epoch", epoch, "| best validation PR-AUC", round(best_score, 3))
            break

# Put the best weights back into the model.
model.load_state_dict(best_weights)
nn_seconds = time.perf_counter() - start
print(f"training took {nn_seconds:.1f}s")

The `history` list now holds one row per epoch. Turn it into a table and look at the last few epochs.

In [ ]:
history = pd.DataFrame(history)
history.tail()

> **Where we are.** The network has trained, and early stopping kept the weights from its best epoch on the 2024
> validation claims. The `history` table records every epoch. Now you'll read it as a picture.

### Reading the training curves

The most useful picture in deep learning is the **training loss** and **validation loss** plotted together, epoch by
epoch. Here's how to read it:

- **Training loss** usually keeps falling: the network keeps getting better at the claims it's learning from.
- **Validation loss** falls at first (the network is learning real patterns), then flattens or starts to rise.
- The point where the validation loss stops improving is where **overfitting** begins: from there on, the network is
  learning the quirks of the training claims rather than patterns that carry over to new claims. Early stopping keeps
  the weights from around that point.

In [ ]:
# Training and validation loss per epoch.
plt.plot(history["epoch"], history["train_loss"], color="#2a78d6", label="train loss")
plt.plot(history["epoch"], history["val_loss"], color="#eb6834", label="validation loss")
plt.title("Loss")
plt.xlabel("epoch")
plt.legend()
plt.show()

And the measure early stopping watched: validation PR-AUC, epoch by epoch. Higher is better.

In [ ]:
# Validation PR-AUC per epoch.
plt.plot(history["epoch"], history["val_pr_auc"], color="#1baf7a")
plt.title("Validation PR-AUC")
plt.xlabel("epoch")
plt.show()

Read the curves. Training loss keeps falling, because the network keeps getting better at the data it
sees. Validation loss falls, then flattens or turns up. That turning point is where the model starts learning
the quirks of the training claims rather than patterns that generalise. Early stopping kept the weights from
around that point.

**Your turn (1).** Make the network much bigger: hidden layers of 256 and 128 neurons, and no dropout. Retrain it
(you'll need a new model *and* a new optimiser, then the training-loop cell again). What happens to the gap between
training and validation loss, and to the best validation PR-AUC (area under the precision-recall curve, the better
guide when positives are rare)?

In [ ]:
# Your code here

> **Where we are.** You've built, trained and examined a neural network, and seen how a bigger network overfits. Now
> the question the lab started with.

### Theory: what "honest" means in a model comparison

It's tempting to give the new model every advantage: tune it carefully, check it many times, report its best run. That
makes any new model look good. An honest comparison gives every model the **same chances**:

- the same test claims, looked at once;
- the same measures, including the one the business cares about (5% capacity);
- each model trained on all the data it can properly use. The network needs a validation set to decide when to stop,
  so it trains on 2019-2023; the others don't, so they get 2019-2024;
- the costs reported too: training time, code, settings, and how hard it is to explain.

And then the result is reported as it is, even when the new, exciting model doesn't win.

## 7. The honest comparison

Now the question the lab started with: is the neural network better than Day 2's models? To answer it fairly:

- every model is tested on the same 2025 claims, which none of them has seen;
- every model is measured in the same way, including the Day 2 "5% capacity" measure (if the review team checks the
  top 5% of claims by score, what share are irregular, and what share of all irregular claims do they catch?).

Two small functions first. `predict_scores` runs claims through the network and turns the raw scores into probabilities.
`at_capacity` is the Day 2 capacity measure: sort by score, take the top 5%, and count.

In [ ]:
def predict_scores(model, X):
    """Score a feature array with the network: logits -> probabilities between 0 and 1."""
    model.eval()                                       # dropout off
    with torch.no_grad():
        logits = model(torch.from_numpy(X).to(device)).squeeze(1)
        return torch.sigmoid(logits).cpu().numpy()     # sigmoid turns raw scores into probabilities


def at_capacity(y_true, risk_scores, capacity=0.05):
    """Precision and recall if the team reviews the top 5% of claims by score (as on Day 2)."""
    # Put each claim's answer next to its score, riskiest first.
    ranked = pd.DataFrame({"irregular": y_true, "score": risk_scores})
    ranked = ranked.sort_values("score", ascending=False)
    # The top 5% are the ones checked. Count the irregular claims among them.
    n_review = round(capacity * len(ranked))
    caught = ranked.head(n_review)["irregular"].sum()
    # precision = share of checked claims that are irregular; recall = share of all irregular claims caught.
    return caught / n_review, caught / ranked["irregular"].sum()

In [ ]:
# The network's scores (probabilities) for the 2025 test claims. The first five:
p_nn = predict_scores(model, X_te)
p_nn[:5]

The classical models (logistic regression and XGBoost) don't need a validation set to decide when to stop, so it would
be unfair to give them less data than they could use. They train on 2019-2024: the training and validation claims
together. `np.vstack` stacks two tables on top of each other; `np.concatenate` joins two lists.

In [ ]:
# Stack the training and validation rows on top of each other.
X_trva = np.vstack([X_tr, X_va])
y_trva = np.concatenate([y_tr, y_va])
print(X_trva.shape)

**Logistic regression**, as on Day 2: one weight per input, `class_weight="balanced"` to take rare irregular claims
seriously. The training time is measured too, because it's a real cost.

In [ ]:
# Logistic regression.
start = time.perf_counter()
lr_model = LogisticRegression(max_iter=2000, class_weight="balanced")
lr_model.fit(X_trva, y_trva)
lr_seconds = time.perf_counter() - start

# Probability of "irregular" for each test claim.
p_lr = lr_model.predict_proba(X_te)[:, 1]

**XGBoost**, with exactly the Day 2 settings (400 small trees, each at most 4 questions deep, small learning steps).
`scale_pos_weight` counts each irregular claim about 10 times, like `pos_weight` for the network.

In [ ]:
# XGBoost, with the same settings as Day 2.
spw = (y_trva == 0).sum() / (y_trva == 1).sum()

start = time.perf_counter()
xgb_model = XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                          scale_pos_weight=spw, eval_metric="logloss", random_state=SEED, n_jobs=4)
xgb_model.fit(X_trva, y_trva)
xgb_seconds = time.perf_counter() - start

p_xgb = xgb_model.predict_proba(X_te)[:, 1]

Now one table with one row per model:

- **roc_auc**: how well the scores put irregular claims above regular ones overall (0.5 = guessing, 1 = perfect);
- **pr_auc**: the same, paying most attention to the top of the list;
- **precision_at_5pct / recall_at_5pct**: the review team's view, at 5% capacity;
- **fit_seconds**: how long training took.

In [ ]:
# One row per model.
rows = []
for name, scores, seconds in [("Logistic regression", p_lr, lr_seconds),
                              ("XGBoost", p_xgb, xgb_seconds),
                              ("Neural network", p_nn, nn_seconds)]:
    precision, recall = at_capacity(y_te, scores)
    rows.append({
        "model": name,
        "roc_auc": roc_auc_score(y_te, scores),
        "pr_auc": average_precision_score(y_te, scores),
        "precision_at_5pct": precision,
        "recall_at_5pct": recall,
        "fit_seconds": seconds,
    })

comparison = pd.DataFrame(rows).set_index("model")
comparison.round(3)

**Did the neural network add value?** Be honest about what the table says. On tabular data of this size
(a few thousand rows, a few dozen features), a small network typically lands somewhere near logistic
regression and below a well-tuned gradient-boosted model. It also needed more code, more choices
(layers, learning rate, dropout, epochs), a validation set, and more care to reproduce.

That's the lesson, and it's an important one for an enterprise team: **don't choose deep learning because
it's more sophisticated.** Choose it where it earns its place: images (scanned claim documents, ID
verification), text (Day 6), sound, very large datasets with complex interactions, or when you need to learn
representations to reuse elsewhere. For a table of claims features, the boosted trees from Day 2 remain the
sensible choice.

**Your turn (2).** Neural networks are sensitive to random initialisation. Train the network three times with
seeds 1, 2 and 3 (call `torch.manual_seed(seed)` before creating the model) and record the test PR-AUC each
time. How much does it vary? What does that mean for comparing it with XGBoost on a single run?

In [ ]:
# Your code here

> **Where we are.** On this table of claims, the network didn't beat Day 2's boosted trees, and it cost more code,
> more settings and more care. That's a real and useful result. Even so, it's worth learning how to save and reload a
> network properly, because you'll do it with every deep-learning model from Day 6 onwards.

## 8. Save and reload

A trained model is only useful if it can leave the notebook and be used by another program (for example, an API that
scores new claims, as on Day 10).

For PyTorch the recommended way is to save the **state dict**: a dictionary of all the learned weights, by name. You
don't save the whole Python object, because that ties the file to the exact code and library versions that created it.
But weights alone aren't enough to use the model. Whoever loads it also needs:

1. **the weights** (`.pt` file);
2. **the fitted preparation steps**: the medians, scaling and category lists learned in Section 2. New claims must be
   prepared exactly the same way, or the network gets inputs it was never trained on;
3. **the settings** needed to rebuild an empty network of the right shape, and the list of input columns.

In [ ]:
Path("models").mkdir(exist_ok=True)          # create the folder if it doesn't exist

# 1. The learned weights.
torch.save(model.state_dict(), "models/risknet_weights.pt")

# 2. The fitted preparation steps (medians, scaling, category lists).
joblib.dump(prep, "models/risknet_preprocessor.joblib")

# 3. The settings needed to rebuild the network, and the columns it expects.
config = {
    "n_inputs": int(n_inputs),
    "hidden": [64, 32],
    "dropout": 0.2,
    "numeric": NUMERIC,
    "categorical": CATEGORICAL,
    "trained_to": "2023-12-31",
}
Path("models/risknet_config.json").write_text(json.dumps(config, indent=2))
print("saved 3 files in models/")

Now pretend you're a different program, later, that has only those three files. To use the model it must:

1. read the settings;
2. build an **empty** network with exactly the same layers;
3. load the saved weights into it.

In [ ]:
# Read the settings.
config = json.loads(Path("models/risknet_config.json").read_text())

# Rebuild an empty network with the same shape...
reloaded = nn.Sequential(
    nn.Linear(config["n_inputs"], config["hidden"][0]),
    nn.ReLU(),
    nn.Dropout(config["dropout"]),
    nn.Linear(config["hidden"][0], config["hidden"][1]),
    nn.ReLU(),
    nn.Linear(config["hidden"][1], 1),
)

# ...and load the saved weights into it. map_location="cpu" lets a file saved on a GPU load on any machine.
reloaded.load_state_dict(torch.load("models/risknet_weights.pt", map_location="cpu"))
reloaded = reloaded.to(device)

The proof that saving worked: prepare five test claims with the **saved** preparation steps, score them with the
**reloaded** network, and check you get the same scores as the original network.

In [ ]:
# Prepare five claims with the saved preprocessor and score them.
prep_reloaded = joblib.load("models/risknet_preprocessor.joblib")
new_claims = test.head(5)
X_new = prep_reloaded.transform(new_claims[config["numeric"] + config["categorical"]]).astype(np.float32)

scores_reloaded = predict_scores(reloaded, X_new)
scores_original = predict_scores(model, X_te[:5])
print("reloaded:", scores_reloaded.round(4))
print("original:", scores_original.round(4))

# np.allclose: True if the two lists match (allowing for tiny rounding differences).
print("same predictions after reload:", np.allclose(scores_reloaded, scores_original))

> **Where we are.** The network is saved as three files (weights, preparation steps, settings), and you've proved the
> reloaded copy gives identical scores. Finally, a look at the other kinds of neural network and where each would be
> useful at the Fund.

Three files, one model. Lose any one of them and the other two are useless, which is why tools like
MLflow (Day 10) package them together.

## 9. Where do CNNs and RNNs fit?

The network you built is a **feed-forward** network: each claim is a fixed list of numbers that flows straight through
the layers. Other families of networks are designed for other shapes of data. You won't build them today, but you
should know where they'd earn their place at the Fund:

| Architecture | What it's designed for | Possible Fund use |
|---|---|---|
| Feed-forward (today) | A fixed list of numbers per example | Rarely better than boosted trees on tables of data |
| CNN (convolutional neural network) | Images: it looks for small local patterns (edges, shapes, characters) anywhere in the picture | Reading scanned claim forms, checking ID photos, spotting tampered documents |
| RNN / LSTM (recurrent network) | Sequences where order matters: it reads one step at a time and remembers what came before | A member's month-by-month contribution history; now largely replaced by transformers |
| Transformer | Text and long sequences: every word can look at every other word at once | Member feedback (Day 6), large language models (Days 7-9) |

TensorFlow/Keras is another library that does the same jobs as PyTorch. The concepts you learned today (layers, loss,
optimiser, training loop, validation) carry over directly. Keras hides the loop inside `model.fit()`; PyTorch makes you
write it, which is why it's the better one to learn on.

## Looking back, looking ahead

**What you did in this lab.** You saw one neuron learn by hand, then built and trained a full network in PyTorch:
tensors, a `DataLoader` of mini-batches, layers with ReLU and dropout, a weighted loss, the Adam optimiser, a training
loop with early stopping, and training curves to read. Then you compared it honestly with Day 2's models, and saved it
so another program could use it.

**The ideas to carry forward.**

- Every neural network, however large, learns with the same loop: predict, loss, gradients, step.
- The validation set is what tells you when to stop.
- On a table of well-engineered features, boosted trees are usually the better choice. Deep learning earns its place on
  images, sound, text and very large data.
- A model is only usable with its preparation steps and settings saved alongside it.

**What happens next.** Lab 2 moves from modelling to scale: building features from about 2.4 million contribution
records without running out of memory.

---

## Answers to the "Your turn" exercises

Both exercises retrain the network, so first wrap the training-loop cell from Section 6 in a function. The body is
the same code; the only change is that the model, optimiser, epochs and patience come in as inputs. A function lets you
run the same loop several times without copying it.

In [ ]:
def train_network(model, optimiser, epochs=60, patience=8):
    """The Section 6 training loop as a function. Returns the history table; the model ends with its best weights."""
    history = []
    best_score = -1
    best_weights = None
    epochs_without_improvement = 0

    for epoch in range(1, epochs + 1):
        # 1. Learn from every mini-batch.
        model.train()
        total_loss = 0.0
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb).squeeze(1)
            loss = loss_fn(logits, yb)
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
            total_loss = total_loss + loss.item() * len(yb)
        train_loss = total_loss / len(train_dataset)

        # 2. Check the validation set.
        model.eval()
        with torch.no_grad():
            val_logits = model(X_va_t).squeeze(1)
            val_loss = loss_fn(val_logits, y_va_t).item()
            val_scores = torch.sigmoid(val_logits).cpu().numpy()
        val_pr_auc = average_precision_score(y_va, val_scores)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_pr_auc": val_pr_auc})

        # 3. Early stopping.
        if val_pr_auc > best_score:
            best_score = val_pr_auc
            best_weights = {}
            for name, value in model.state_dict().items():
                best_weights[name] = value.clone()
            epochs_without_improvement = 0
        else:
            epochs_without_improvement = epochs_without_improvement + 1
            if epochs_without_improvement >= patience:
                break

    model.load_state_dict(best_weights)
    return pd.DataFrame(history)

**(1)** A bigger network without dropout: hidden layers of 256 and 128 instead of 64 and 32 (about ten times as many
weights), and no `Dropout` layer.

In [ ]:
torch.manual_seed(SEED)
big_model = nn.Sequential(
    nn.Linear(n_inputs, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 1),
).to(device)
big_optimiser = torch.optim.Adam(big_model.parameters(), lr=1e-3)

# patience=40 with 40 epochs: no early stopping, so you see the whole curve.
big_history = train_network(big_model, big_optimiser, epochs=40, patience=40)

In [ ]:
# The same loss chart as Section 6, for the big network.
plt.plot(big_history["epoch"], big_history["train_loss"], color="#2a78d6", label="train loss")
plt.plot(big_history["epoch"], big_history["val_loss"], color="#eb6834", label="validation loss")
plt.title("Big network, no dropout")
plt.xlabel("epoch")
plt.legend()
plt.show()
print("best validation PR-AUC:", round(big_history["val_pr_auc"].max(), 3))

Training loss keeps dropping while validation loss climbs: classic overfitting. The bigger network memorises
the training claims. It has far more parameters than there are irregular claims to learn from.

**(2)** Seed variation: train the same small network three times, changing only the seed (which changes the random
starting weights and the order of the batches), and compare the test PR-AUC.

In [ ]:
test_pr_aucs = []

for seed in [1, 2, 3]:
    torch.manual_seed(seed)                     # a different starting point each time
    seed_model = nn.Sequential(
        nn.Linear(n_inputs, 64), nn.ReLU(), nn.Dropout(0.2),
        nn.Linear(64, 32), nn.ReLU(),
        nn.Linear(32, 1),
    ).to(device)
    seed_optimiser = torch.optim.Adam(seed_model.parameters(), lr=1e-3, weight_decay=1e-4)
    train_network(seed_model, seed_optimiser)

    # Score the test claims and measure PR-AUC.
    pr_auc = average_precision_score(y_te, predict_scores(seed_model, X_te))
    test_pr_aucs.append(pr_auc)
    print("seed", seed, "test PR-AUC:", round(pr_auc, 3))

print("spread (max - min):", round(max(test_pr_aucs) - min(test_pr_aucs), 3))

If the spread between seeds is similar in size to the gap between the network and another model, then a
single run can't tell you which is better. Report an average over several seeds, or better, use the
year-by-year backtest from Day 2.